In [5]:
import sys
!{sys.executable} -m pip install pandas numpy torch scikit-learn

  Obtaining dependency information for pandas from https://files.pythonhosted.org/packages/35/d0/4831af68ce30cc2d03c697bea8450e3225a835ef497d0d70f31b8cdde965/pandas-3.0.2-cp312-cp312-macosx_11_0_arm64.whl.metadata
  Using cached pandas-3.0.2-cp312-cp312-macosx_11_0_arm64.whl.metadata (79 kB)
  Obtaining dependency information for numpy from https://files.pythonhosted.org/packages/9b/fd/e5ecca1e78c05106d98028114f5c00d3eddb41207686b2b7de3e477b0e22/numpy-2.4.4-cp312-cp312-macosx_14_0_arm64.whl.metadata
  Using cached numpy-2.4.4-cp312-cp312-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Obtaining dependency information for torch from https://files.pythonhosted.org/packages/6f/8b/69e3008d78e5cee2b30183340cc425081b78afc5eff3d080daab0adda9aa/torch-2.11.0-cp312-cp312-macosx_11_0_arm64.whl.metadata
  Using cached torch-2.11.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (29 kB)
  Obtaining dependency information for scikit-learn from https://files.pythonhosted.org/packages/49/d8/9be608c6024d021041c7f

In [6]:
import sys
!{sys.executable} -m pip install pandas numpy torch scikit-learn


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [7]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import LabelEncoder
print("All libraries loaded! ✅")

All libraries loaded! ✅


In [8]:
# Sample order data (MySQL-லிருந்து later real data வரும்)
data = {
    'user_id':    [1, 1, 2, 2, 3, 3, 4, 4, 5],
    'product_id': [101, 102, 101, 103, 102, 104, 103, 101, 104],
    'rating':     [5, 4, 5, 3, 4, 5, 4, 3, 5]
}
df = pd.DataFrame(data)
print(df)

   user_id  product_id  rating
0        1         101       5
1        1         102       4
2        2         101       5
3        2         103       3
4        3         102       4
5        3         104       5
6        4         103       4
7        4         101       3
8        5         104       5


In [9]:
def get_recommendations(user_id, df, top_n=3):
    # இந்த user என்ன books பாத்தாங்க
    user_books = df[df['user_id'] == user_id]['product_id'].tolist()
    print(f"User {user_id} already saw: {user_books}")
    
    # அந்த books தவிர மத்தவை
    other_books = df[~df['product_id'].isin(user_books)]
    
    # Average rating வச்சு sort பண்ணு
    recommendations = (
        other_books.groupby('product_id')['rating']
        .mean()
        .nlargest(top_n)
        .index.tolist()
    )
    return recommendations

# Test பண்ணு
result = get_recommendations(user_id=1, df=df)
print(f"Recommended product IDs: {result}")

User 1 already saw: [101, 102]
Recommended product IDs: [104, 103]


In [10]:
class BookRecommender(nn.Module):
    def __init__(self, num_users, num_books, embedding_dim=8):
        super().__init__()
        # User-ku ஒரு embedding vector
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        # Book-ku ஒரு embedding vector
        self.book_embedding = nn.Embedding(num_books, embedding_dim)
    
    def forward(self, user_id, book_id):
        user_vec = self.user_embedding(user_id)
        book_vec = self.book_embedding(book_id)
        # Dot product = similarity score
        score = (user_vec * book_vec).sum(dim=1)
        return score

# Model create பண்ணு
model = BookRecommender(num_users=10, num_books=200)
print("PyTorch Model ready! ✅")
print(model)

PyTorch Model ready! ✅
BookRecommender(
  (user_embedding): Embedding(10, 8)
  (book_embedding): Embedding(200, 8)
)


In [11]:
# Model save பண்ணு — later FastAPI use பண்ணும்
torch.save(model.state_dict(), "book_recommender_model.pt")
print("Model saved as book_recommender_model.pt ✅")

Model saved as book_recommender_model.pt ✅
